# Comparing DBSCAN and HDBSCAN clustering 

## Objectives

After completing this lab, you will be able to:

* Use scikit-learn to implement DBSCAN and HDBSCAN clustering models to real data
* Compare the performances of the two models 



## Introduction
In this lab, you'll create two clustering models using data curated by StatCan containing the names, types, and locations of
cultural and art facilities across Canada.
We'll focus on the museum locations provided across Canada.

#### Data source: The Open Database of Cultural and Art Facilities (ODCAF)

A collection of open data containing the names, types, and locations of cultural and art facilities across Canada. 
It is released under the Open Government License - Canada.
The different types of facilities are labeled under 'ODCAF_Facility_Type'.

#### Landing page:
https://www.statcan.gc.ca/en/lode/databases/odcaf

#### link to zip file:
https://www150.statcan.gc.ca/n1/en/pub/21-26-0001/2020001/ODCAF_V1.0.zip?st=brOCT3Ry


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import requests


In [3]:
def download (url, fileName):
    response = requests.get(url)
    if(response.status_code == 200):
        with open(fileName, "wb") as f:
            f.write(response.content)

In [5]:
file_path =  'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/r-maSj5Yegvw2sJraT15FA/ODCAF-v1-0.csv'
file_name = 'ODCAF-v1-0.csv'

In [5]:
download(file_path,file_name)

In [6]:
df = pd.read_csv(file_name, encoding='ISO-8859-1')
df.head()

,Index,Facility_Name,Source_Facility_Type,ODCAF_Facility_Type,Provider,Unit,Street_No,Street_Name,Postal_Code,City,Prov_Terr,Source_Format_Address,CSD_Name,CSDUID,PRUID,Latitude,Longitude
0,1,#Hashtag Gallery,..,gallery,toronto,..,801,dundas st w,M6J 1V2,toronto,on,801 dundas st w,Toronto,3520005,35,43.65169472,-79.40803272
1,2,'Ksan Historical Village & Museum,historic site-building or park,museum,canadian museums association,..,1500,62 hwy,V0J 1Y0,hazelton,bc,1500 hwy 62 hazelton british columbia v0j 1y0 ...,Hazelton,5949022,59,55.2645508,-127.6428124
2,3,'School Days' Museum,community/regional museum,museum,canadian museums association,..,427,queen st,E3B 5R6,fredericton,nb,427 queen st fredericton new brunswick e3b 5r6...,Fredericton,1310032,13,45.963283,-66.6419017
3,4,10 Austin Street,built heritage properties,heritage or historic site,moncton,..,10,austin st,E1C 1Z6,moncton,nb,10 austin st,Moncton,1307022,13,46.09247776,-64.78022946
4,5,10 Gates Dancing Inc.,arts,miscellaneous,ottawa,..,..,..,..,ottawa,on,..,Ottawa,3506008,35,45.40856224,-75.71536766


## Data Preprocessing


In [7]:
df.isnull().sum()

Index                    0
Facility_Name            0
Source_Facility_Type     0
ODCAF_Facility_Type      0
Provider                 0
Unit                     0
Street_No                0
Street_Name              0
Postal_Code              0
City                     0
Prov_Terr                0
Source_Format_Address    0
CSD_Name                 0
CSDUID                   0
PRUID                    0
Latitude                 0
Longitude                0
dtype: int64

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7972 entries, 0 to 7971
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Index                  7972 non-null   int64
 1   Facility_Name          7972 non-null   str  
 2   Source_Facility_Type   7972 non-null   str  
 3   ODCAF_Facility_Type    7972 non-null   str  
 4   Provider               7972 non-null   str  
 5   Unit                   7972 non-null   str  
 6   Street_No              7972 non-null   str  
 7   Street_Name            7972 non-null   str  
 8   Postal_Code            7972 non-null   str  
 9   City                   7972 non-null   str  
 10  Prov_Terr              7972 non-null   str  
 11  Source_Format_Address  7972 non-null   str  
 12  CSD_Name               7972 non-null   str  
 13  CSDUID                 7972 non-null   str  
 14  PRUID                  7972 non-null   str  
 15  Latitude               7972 non-null   str  
 16 

In [12]:
# Display ODCAF_Facility_Type and their counts

df['ODCAF_Facility_Type'].value_counts()

ODCAF_Facility_Type
library or archives                     3013
museum                                  1938
gallery                                  810
heritage or historic site                620
theatre/performance and concert hall     583
festival site                            346
miscellaneous                            343
art or cultural centre                   225
artist                                    94
Name: count, dtype: int64

In [17]:
#Filter the data to only include museums.
df_mus = df[df['ODCAF_Facility_Type'] == 'museum']
df_mus 

,Index,Facility_Name,Source_Facility_Type,ODCAF_Facility_Type,Provider,Unit,Street_No,Street_Name,Postal_Code,City,Prov_Terr,Source_Format_Address,CSD_Name,CSDUID,PRUID,Latitude,Longitude
1,2,'Ksan Historical Village & Museum,historic site-building or park,museum,canadian museums association,..,1500,62 hwy,V0J 1Y0,hazelton,bc,1500 hwy 62 hazelton british columbia v0j 1y0 ...,Hazelton,5949022,59,55.2645508,-127.6428124
2,3,'School Days' Museum,community/regional museum,museum,canadian museums association,..,427,queen st,E3B 5R6,fredericton,nb,427 queen st fredericton new brunswick e3b 5r6...,Fredericton,1310032,13,45.963283,-66.6419017
8,10,12 Service Battalion Museum,military museum or fort,museum,canadian museums association,..,5500,no 4 rd,V6X 3L5,richmond,bc,5500 no. 4 rd the sherman armoury richmond bri...,Richmond,5915015,59,49.1763542,-123.112783
13,15,15th Field Artillery Regiment Museum And Archives,museum/gallery,museum,vancouver,..,2025,11th av w,V6J 2C7,vancouver,bc,2025 w 11th av vancouver bc v6j 2c7,Vancouver,5915022,59,49.261938,-123.151123
15,18,17 Wing Heritage Collection,aeronautics and space museum transportation mu...,museum,canadian museums association,..,..,..,R3J 3Y5,winnipeg,mb,air heritage park air force way winnipeg manit...,Winnipeg,4611040,46,49.88955855,-97.23574396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7940,9765,York Region District School Board Museum & Arc...,library and/or archives community/regional museum,museum,canadian museums association,..,21,renfrew dr,L3R 8H3,markham,on,21 renfrew dr markham ontario l3r 8h3 canada,Markham,3519036,35,43.857692,-79.3619396
7954,9779,Youthlink Calgary-The Calgary Police Interpret...,community/regional museum exhibition or cultur...,museum,canadian museums association,..,5111,47th street ne,T3J 3R2,calgary,ab,5111-47th street ne lc594 calgary alberta t3j ...,Calgary,4806016,48,..,..
7958,9783,Yukon Beringia Interpretive Centre,human history-archaelogy-anthropology or ethno...,museum,canadian museums association,..,1423,alaska hwy,Y1A 2C6,whitehorse,yt,kilometre 1423 (mile 886) alaska hwy po box 27...,Whitehorse,6001009,60,..,..
7968,9797,Craigdarroch Castle,museums and galleries,museum,victoria,..,..,..,..,victoria,bc,..,Victoria,5917034,59,48.42241956,-123.3435527


Select only the Latitude and Longitude features as inputs to our clustering problem.
Also, display information about the coordinates like counts and data types.

In [35]:
df_data = df_mus.filter(['Latitude', 'Longitude'], axis=1)
df_data

,Latitude,Longitude
1,55.2645508,-127.6428124
2,45.963283,-66.6419017
8,49.1763542,-123.112783
13,49.261938,-123.151123
15,49.88955855,-97.23574396
...,...,...
7940,43.857692,-79.3619396
7954,..,..
7958,..,..
7968,48.42241956,-123.3435527


In [27]:
df_data.info()

<class 'pandas.DataFrame'>
Index: 1938 entries, 1 to 7969
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Latitude   1938 non-null   str  
 1   Longitude  1938 non-null   str  
dtypes: str(2)
memory usage: 45.4 KB


In [ ]:
# Remove observations with no coordinates 
df_data = df_data[df_data['Latitude'] != '..']

df_data

,Latitude,Longitude
1,55.2645508,-127.6428124
2,45.963283,-66.6419017
8,49.1763542,-123.112783
13,49.261938,-123.151123
15,49.88955855,-97.23574396
...,...,...
7934,43.18308983,-79.2245641
7936,43.6900216,-79.476208
7940,43.857692,-79.3619396
7968,48.42241956,-123.3435527


In [45]:
df_data = df_data[df_data['Longitude'] != '..']
df_data.value_counts()

Latitude     Longitude   
51.01423945  -114.1162943    6
45.71061766  -63.2859451     3
48.369878    -53.359843      3
49.89093415  -97.153417      2
49.0960503   -99.3400468     2
                            ..
43.18308983  -79.2245641     1
43.6900216   -79.476208      1
43.857692    -79.3619396     1
48.42241956  -123.3435527    1
48.4260053   -123.3691883    1
Name: count, Length: 1583, dtype: int64

In [47]:
# Remove rows where Latitude or Longitude is ".."
df_data = df_data[(df_data['Latitude'] != "..") & (df_data['Longitude'] != "..")]

# Convert columns into floats
df_data[['Latitude', 'Longitude']] = df_data[['Latitude', 'Longitude']].astype('float64')

In [48]:
df_data.info()

<class 'pandas.DataFrame'>
Index: 1607 entries, 1 to 7969
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Latitude   1607 non-null   float64
 1   Longitude  1607 non-null   float64
dtypes: float64(2)
memory usage: 37.7 KB
